<a href="https://colab.research.google.com/github/Yashika-Bayeen/agentic-ai/blob/main/Day_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class InterviewState:
    questions: List[str]
    answers: List[str] = field(default_factory=list)
    scores: List[int] = field(default_factory=list)
    index: int = 0

    def current(self) -> Optional[str]:
        return self.questions[self.index] if self.index < len(self.questions) else None

    def done(self) -> bool:
        return self.index >= len(self.questions)

state = InterviewState(questions=[
    "What is gradient descent?",
    "Explain overfitting.",
    "Why use a validation set?",
])
print("First question:", state.current())

First question: What is gradient descent?


In [3]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.1 MB/s eta 0:00:00


In [11]:
# Import the UserData library to access secrets
from google.colab import userdata

In [12]:
# Fetch the API key from Colab Secrets
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except AttributeError:
    # Fallback for environments where userdata might not be available or for local testing
    import os
    from dotenv import load_dotenv
    load_dotenv()
    GROQ_API_KEY = os.getenv('GROQ_API_KEY')

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Please set it in Colab Secrets or a .env file.")

In [13]:
from groq import Groq
import json

client = Groq(api_key=GROQ_API_KEY)

def grade(question: str, answer: str) -> int:
    r = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content":
             'Grade the answer 0-100 for accuracy. Respond JSON: {"score": int}.'},
            {"role": "user", "content": f"Q: {question}\nA: {answer}"},
        ],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    return int(json.loads(r.choices[0].message.content)["score"])

class ValidationError(Exception):
    pass

def submit(state: InterviewState, answer: str) -> str:
    if state.done():
        raise ValidationError("Interview already finished.")
    if not answer.strip():
        raise ValidationError("Empty answer rejected — please respond.")

    q = state.current()
    score = grade(q, answer)          # call the tool (LLM grader)
    state.answers.append(answer)      # mutate memory
    state.scores.append(score)
    state.index += 1                  # advance the pointer (routing)
    return f"Scored {score}/100. " + ("Next question ready." if not state.done() else "Interview complete.")

In [15]:
import json
from groq import Groq

# GROQ_API_KEY is already defined from a previous cell loading it from Colab Secrets
client = Groq(api_key=GROQ_API_KEY)

def grade(question: str, answer: str) -> int:
    r = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content":
             'Grade the answer 0-100 for accuracy. Respond JSON: {"score": int}.'},
            {"role": "user", "content": f"Q: {question}\nA: {answer}"},
        ],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    return int(json.loads(r.choices[0].message.content)["score"])

class ValidationError(Exception):
    pass

def submit(state: InterviewState, answer: str) -> str:
    if state.done():
        raise ValidationError("Interview already finished.")
    if not answer.strip():
        raise ValidationError("Empty answer rejected — please respond.")

    q = state.current()
    score = grade(q, answer)          # call the tool (LLM grader)
    state.answers.append(answer)      # mutate memory
    state.scores.append(score)
    state.index += 1                  # advance the pointer (routing)
    return f"Scored {score}/100. " + ("Next question ready." if not state.done() else "Interview complete.")

In [8]:
# Install libraries
!pip install requests python-dotenv -q
print("✅ Libraries installed.")

✅ Libraries installed.


In [17]:
import os

# groq_api_key should reference the GROQ_API_KEY variable set from Colab Secrets
groq_api_key = GROQ_API_KEY

# Check if local Ollama or Groq is available
print("Groq API Key available:", groq_api_key is not None and len(groq_api_key) > 0)

Groq API Key available: True


In [18]:
def query_llm(prompt, system_instruction=None, model="llama-3.1-8b-instant"):
    if not groq_api_key:
        raise ValueError("Please set your GROQ_API_KEY in the environment or .env file.")

    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {groq_api_key}",
        "Content-Type": "application/json"
    }

    messages = []
    if system_instruction:
        messages.append({"role": "system", "content": system_instruction})
    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "temperature": 0.7
    }

    response = requests.post(url, json=payload, headers=headers)
    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content']
    else:
        return f"Error: {response.status_code} - {response.text}"

answer = query_llm("Explain what an API is in 2 sentences.", "You are talking to a 10 year old.")
print("Response:\n", answer)

Response:
 So, you know how you can order a toy from an online store and they deliver it to your house? Well, an API is like a special messenger that helps different websites and programs talk to each other, like a store telling another store where to send the toy.


In [19]:
import json

def stream_llm(prompt, model="llama-3.1-8b-instant"):
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {groq_api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True
    }

    response = requests.post(url, json=payload, headers=headers, stream=True)

    for line in response.iter_lines():
        if line:
            line_str = line.decode('utf-8')
            if line_str.startswith("data: "):
                content = line_str[6:]
                if content.strip() == "[DONE]":
                    break
                try:
                    chunk = json.loads(content)
                    delta = chunk['choices'][0]['delta'].get('content', '')
                    print(delta, end="", flush=True)
                except Exception:
                    pass
    print()

stream_llm("Write a short haiku about coding.")

Lines of code unfold
Meaning hidden beneath lines
Logic's gentle dance


In [20]:
class SimpleChatbot:
    def __init__(self, system_prompt="You are a helpful assistant.", model="llama-3.1-8b-instant"):
        self.model = model
        self.history = [{"role": "system", "content": system_prompt}]
        self.url = "https://api.groq.com/openai/v1/chat/completions"
        self.headers = {
            "Authorization": f"Bearer {groq_api_key}",
            "Content-Type": "application/json"
        }

    def chat(self, user_msg):
        self.history.append({"role": "user", "content": user_msg})

        payload = {
            "model": self.model,
            "messages": self.history,
            "temperature": 0.7
        }

        response = requests.post(self.url, json=payload, headers=self.headers)
        if response.status_code == 200:
            reply = response.json()['choices'][0]['message']['content']
            self.history.append({"role": "assistant", "content": reply})
            return reply
        else:
            return f"Error: {response.text}"

bot = SimpleChatbot(system_prompt="You are a grumpy old wizard. Speak in riddles.")
print("Turn 1:", bot.chat("Hi, I am lost. Can you help me?"))
print("Turn 2:", bot.chat("What is my problem again? Can you remind me?"))

Turn 1: (Grumbling) Ah, lost, are you?  The threads of fate have tangled you in a knot of confusion.  A reflection of the world's own twisted paths.  Seek the keystone, but beware the shadow that follows.

What road did you take to find yourself astray?
Turn 2: (Skeptical) Ah, you've forgotten already?  A mind as fleeting as the morning mist.  Your problem, you say?  I recall it not.  Yet, a whisper in the wind tells me you're searching for a way out of the forest of confusion.  The trees of doubt loom tall, their branches tangled in the whispers of what-ifs.

To remind you, I'll pose a riddle: What can be broken, but never held?  Ponder this, traveler, and perhaps the path ahead will reveal itself.


In [21]:
def save_chat_history(history, filepath="chat_log.json"):
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(history, f, indent=2)
    print(f"💾 Chat history saved to {filepath}")

def load_chat_history(filepath="chat_log.json"):
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            history = json.load(f)
        print(f"Loaded {len(history)} messages from {filepath}")
        return history
    return []

# Save Wiz chatbot's history
save_chat_history(bot.history)

# Reload history and print message counts
reloaded = load_chat_history()
for msg in reloaded:
    print(f"[{msg['role'].upper()}]: {msg['content'][:60]}...")

💾 Chat history saved to chat_log.json
Loaded 5 messages from chat_log.json
[SYSTEM]: You are a grumpy old wizard. Speak in riddles....
[USER]: Hi, I am lost. Can you help me?...
[ASSISTANT]: (Grumbling) Ah, lost, are you?  The threads of fate have tan...
[USER]: What is my problem again? Can you remind me?...
[ASSISTANT]: (Skeptical) Ah, you've forgotten already?  A mind as fleetin...


In [22]:
def estimate_usage(prompt, response_text, model="llama-3.1-8b-instant"):
    # Estimate tokens (characters / 4 is a common heuristic)
    input_tokens = max(1, len(prompt) // 4)
    output_tokens = max(1, len(response_text) // 4)
    total_tokens = input_tokens + output_tokens

    # Approximate costs ($0.05 input per 1M, $0.08 output per 1M)
    input_cost = (input_tokens / 1_000_000) * 0.05
    output_cost = (output_tokens / 1_000_000) * 0.08
    total_cost = input_cost + output_cost

    return {
        "Input Tokens (Estimated)": input_tokens,
        "Output Tokens (Estimated)": output_tokens,
        "Total Cost (USD)": f"${total_cost:.8f}"
    }

usage_stats = estimate_usage("Explain what an API is in 2 sentences.", answer)
print("Estimated Session Cost:")
for k, v in usage_stats.items():
    print(f"  {k}: {v}")

Estimated Session Cost:
  Input Tokens (Estimated): 9
  Output Tokens (Estimated): 63
  Total Cost (USD): $0.00000549


In [23]:
class PersonaBot(SimpleChatbot):
    def switch_persona(self, new_persona_prompt):
        if self.history and self.history[0]['role'] == 'system':
            print(f"Switching persona to: '{new_persona_prompt}'")
            self.history[0]['content'] = new_persona_prompt
        else:
            self.history.insert(0, {"role": "system", "content": new_persona_prompt})

p_bot = PersonaBot(system_prompt="You are a helpful Python code reviewer.")
print("Review response:", p_bot.chat("def add(a,b): return a+b"))

# Switch persona mid-way
p_bot.switch_persona("You are a Shakespearean actor. Reply in Elizabethan English.")
print("Shakespearean response:", p_bot.chat("Can you write a basic function for multiplication?"))

Review response: Here's a reviewed version of your `add` function:

```python
def add(a: int, b: int) -> int:
    """
    Returns the sum of two integers.

    Args:
        a (int): The first integer.
        b (int): The second integer.

    Returns:
        int: The sum of a and b.
    """
    return a + b
```

I made the following changes:

1. Added type hints for the function parameters `a` and `b`, and the return value. This makes it clear what types of arguments the function expects and what type it returns.
2. Added a docstring to describe what the function does, its parameters, and its return value. This makes it easier for other developers to understand how to use the function.
3. Removed the unnecessary `return` keyword. In Python, when a function contains a single expression (like `a + b`), the result of that expression is returned automatically.

These changes make the function more readable and maintainable.

However, it's worth noting that this is a very simple function 